# Python for Bioinformatics — Part IV Solutions
## Modules 13–18 · Your Own Tools & Pipelines

Reference solutions with commentary.

## Module 13

**13A.1**

In [ ]:
environment_yml = '''
name: biopy
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11
  - biopython
  - pysam
'''
print(environment_yml)

**13A.2**

In [ ]:
# Committing environment.yml lets anyone recreate the exact package set with
# `conda env create -f environment.yml` (reproducible). `--from-history` records
# only the packages you explicitly installed - portable across OSes - rather than
# every transitive dependency pinned to your exact build.

**13B.1**

In [ ]:
# git init
# git add analysis.py
# git commit -m "Add GC-content analysis script"
# git remote add origin https://github.com/USERNAME/mytool.git
# git branch -M main
# git push -u origin main
#
# .gitignore:  __pycache__/   .ipynb_checkpoints/   data/

## Module 14

**14A.1**

In [ ]:
import argparse
def build_parser():
    p = argparse.ArgumentParser(description="Report GC content per sequence.")
    p.add_argument("fasta")
    p.add_argument("--min-gc", type=float, default=0.0)
    p.add_argument("-o", "--output", default=None, help="output file (default stdout)")
    return p
args = build_parser().parse_args(["x.fasta", "-o", "out.txt"])
print(args.fasta, args.min_gc, args.output)

**14A.2**

In [ ]:
gc_tool_py = r"""
import argparse, sys
from Bio import SeqIO

def gc_content(seq):
    seq = str(seq).upper()
    return (seq.count("G") + seq.count("C")) / len(seq)

def main():
    p = argparse.ArgumentParser(description="Print ids with GC >= threshold.")
    p.add_argument("fasta")
    p.add_argument("--min-gc", type=float, default=0.0)
    p.add_argument("-o", "--output", default=None)
    args = p.parse_args()
    out = open(args.output, "w") if args.output else sys.stdout
    for rec in SeqIO.parse(args.fasta, "fasta"):
        gc = gc_content(rec.seq)
        if gc >= args.min_gc:
            print(rec.id, round(gc, 3), sep="\t", file=out)

if __name__ == "__main__":
    main()
"""
print(gc_tool_py)

## Module 15

**15A.1**

In [ ]:
pyproject_toml = r"""
[project]
name = "seqtools"
version = "0.1.0"
description = "Small sequence utilities for bioinformatics"
requires-python = ">=3.10"
dependencies = ["biopython>=1.80"]

[project.scripts]
seq-gc = "seqtools.cli:main"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
"""
print(pyproject_toml)

**15B.1**

In [ ]:
# seqtools/
#  |- pyproject.toml, README.md, LICENSE
#  |- src/seqtools/__init__.py, sequence.py, cli.py
#  |- tests/test_sequence.py
#
# (a) pip install -e .      (b) pytest
# Editable install links to your source, so edits take effect immediately
# without reinstalling - the fast develop/test loop.

## Module 16

**16A.1**

In [ ]:
import subprocess
r = subprocess.run(["python", "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip())

**16A.2**

In [ ]:
import subprocess, sys
def run_step(cmd):
    try:
        return subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"command failed (exit {e.returncode}): {' '.join(cmd)}", file=sys.stderr)
        raise
try:
    run_step(["ls", "no_such_dir_xyz"])
except subprocess.CalledProcessError:
    print("caught the failure as expected")

## Module 17

**17A.1** — needs pysam + an indexed BAM

In [ ]:
code_bam = r"""
import pysam
bam = pysam.AlignmentFile("aln.bam", "rb")
for read in bam.fetch("chr1", 1000, 2000):
    print(read.query_name, read.reference_start, read.mapping_quality)
bam.close()
"""
print(code_bam)
# Region fetch() needs a BAM index aln.bam.bai  (make it: samtools index aln.bam)

**17A.2**

In [ ]:
code_entrez = r"""
from Bio import Entrez, SeqIO
Entrez.email = "you@example.com"   # REQUIRED by NCBI
handle = Entrez.efetch(db="nucleotide", id="NC_000962.3", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()
print(record.id, len(record.seq))
"""
print(code_entrez)
# You must set Entrez.email before any NCBI request.

## Part IV capstone

In [ ]:
snakefile = r"""
SAMPLES = ["rpoB", "katG", "gyrA"]

rule all:
    input: "results/report.txt"

rule gc_report:
    input:  "data/{sample}.fasta"
    output: "results/{sample}.gc.txt"
    shell:  "seq-gc {input} > {output}"

rule combine:
    input:  expand("results/{s}.gc.txt", s=SAMPLES)
    output: "results/report.txt"
    shell:  "cat {input} > {output}"
"""
print(snakefile)
# Run:      snakemake --cores 4
# Dry run:  snakemake --dry-run
# Better than one subprocess script: Snakemake tracks file dependencies and
# reruns only affected steps (incremental), parallelises samples, declares the
# whole workflow in one versioned file (reproducible), and scales laptop->HPC.

---
*You've finished the course* — from first principles to writing, packaging, and shipping your own reproducible tools and pipelines.